In [ ]:
pip install feedparser

In [1]:
import csv
import hashlib
import re
import time
from pathlib import Path
from urllib.parse import urlparse

import feedparser
import requests

RSS_URL = "https://feed.firstory.me/rss/user/ckgt4pmfa3cjt0812r7uko63o"
OUTPUT_DIR = Path("podcast_mp3_all")
LOG_FILE = OUTPUT_DIR / "download_log.csv"
TIMEOUT = 60
SLEEP_BETWEEN_DOWNLOADS = 1.0  # 秒，避免太密集


def safe_filename(name: str) -> str:
    name = re.sub(r'[\\/:*?"<>|]', "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:180]


def get_audio_url(entry) -> str | None:
    enclosures = entry.get("enclosures", [])
    for enc in enclosures:
        href = enc.get("href")
        if href:
            return href

    links = entry.get("links", [])
    for link in links:
        href = link.get("href")
        link_type = link.get("type", "")
        if href and "audio" in link_type:
            return href

    return None


def guess_extension(audio_url: str) -> str:
    path = urlparse(audio_url).path.lower()
    if path.endswith(".mp3"):
        return ".mp3"
    if path.endswith(".m4a"):
        return ".m4a"
    if path.endswith(".aac"):
        return ".aac"
    return ".mp3"


def short_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:10]


def download_file(url: str, output_path: Path) -> None:
    with requests.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 64):
                if chunk:
                    f.write(chunk)


def init_log():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if not LOG_FILE.exists():
        with open(LOG_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["index", "title", "published", "audio_url", "filename", "status", "error"])


def append_log(index, title, published, audio_url, filename, status, error=""):
    with open(LOG_FILE, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([index, title, published, audio_url, filename, status, error])


def main():
    init_log()

    feed = feedparser.parse(RSS_URL)

    if getattr(feed, "bozo", False):
        print("警告：RSS 解析有異常，但仍嘗試繼續。")

    entries = feed.entries
    if not entries:
        print("找不到任何集數。")
        return

    print(f"共找到 {len(entries)} 集，開始下載...")

    success = 0
    skipped = 0
    failed = 0

    for i, entry in enumerate(entries, start=1):
        title = entry.get("title", f"episode_{i}")
        published = entry.get("published", "") or entry.get("updated", "")
        audio_url = get_audio_url(entry)

        if not audio_url:
            print(f"[略過] {title}：找不到音訊連結")
            append_log(i, title, published, "", "", "no_audio_url", "No audio URL found")
            skipped += 1
            continue

        ext = guess_extension(audio_url)
        # 加 hash 避免同名檔案衝突
        filename = safe_filename(f"{i:03d}_{title}_{short_hash(audio_url)}{ext}")
        output_path = OUTPUT_DIR / filename

        if output_path.exists():
            print(f"[已存在] {output_path.name}")
            append_log(i, title, published, audio_url, filename, "skipped_exists", "")
            skipped += 1
            continue

        try:
            print(f"[下載中] {i:03d} {title}")
            download_file(audio_url, output_path)
            print(f"[完成] {output_path.name}")
            append_log(i, title, published, audio_url, filename, "downloaded", "")
            success += 1
        except Exception as e:
            print(f"[失敗] {title} -> {e}")
            append_log(i, title, published, audio_url, filename, "failed", str(e))
            failed += 1

        time.sleep(SLEEP_BETWEEN_DOWNLOADS)

    print("\n===== 結果 =====")
    print(f"成功: {success}")
    print(f"略過: {skipped}")
    print(f"失敗: {failed}")
    print(f"輸出資料夾: {OUTPUT_DIR.resolve()}")
    print(f"紀錄檔: {LOG_FILE.resolve()}")


if __name__ == "__main__":
    main()

共找到 645 集，開始下載...
[已存在] 001_2月營收創高太猛！鈊象甜甜價浮現？_1872a1d25c.mp3
[已存在] 002_狂漲1342點 台股站回3萬4 億元教授_台積隨時可進場｜楚狂人 ft. 億元教授 鄭廳宜｜財富狂犇｜玩股網20260311_8a30831d7e.mp3
[已存在] 003_退休族月領不到2萬_! 主動式基金2招配置養老金｜楚狂人 ft. 大學財金系退休教授 朱岳中｜財富狂犇｜玩股網20260307_79255c2210.mp3
[已存在] 004_EP404｜台美股暴跌，你的幾種應對方式_f6ad1da53f.mp3
[已存在] 005_0056連3季配0.866元 這幾檔持股幫你賺錢 !_0577d7e1cb.mp3
[已存在] 006_台股漲太快見好就收_ 台積電回檔大口吃肉_!｜楚狂人 ft. 資深分析師 許博傑｜財富狂犇｜玩股網20260304_2822cda88c.mp3
[已存在] 007_美股恐慌交易 虛胖_大修正_ 七巨頭抄底好時機來了_｜楚狂人 ft. 美股投資專家 施雅棠｜財富狂犇｜玩股網20260228_f1ef36da9f.mp3
[已存在] 008_EP403｜AI 末日報告炸翻軟體股，你手上的科技股還能抱嗎？_20ce3721b7.mp3
[已存在] 009_富邦金V轉有望？2關鍵拼翻身_5370887431.mp3
[已存在] 010_台積電回檔補票價在這 ! 不怕買高就怕買不夠!｜楚狂人 ft. 資深分析師 陳威良｜財富狂犇｜玩股網20260225_8cc79c9673.mp3
[已存在] 011_EP402｜為什麼我當老闆之後，投資邏輯整個大改變_07a033f0b3.mp3
[已存在] 012_2026美股90%是牛市_！ 七巨頭誰會大反攻｜楚狂人 ft. 美股投資專家 美股航海王｜財富狂犇｜玩股網20260214_0d265308e7.mp3
[已存在] 013_投資天菜是它_! 不怕房市冷清 股息逐年穩定成長_25d6c1bbcb.mp3
[已存在] 014_台股年後紅盤勝率_ 台積賣飛免驚 年後再買回｜楚狂人 ft. 資深分析師 杜金龍｜財富狂犇｜玩股網20260211_66883c0dcb.mp3
[已存在] 015_ETF＋三類股存股法 年領股息衝破百萬｜楚狂人 ft. 投資達